<h2 align="center">Statistical Analysis</h2>

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import (
    pearsonr,
    spearmanr,
    shapiro,
    levene,
    f_oneway,
    kruskal
)
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from pathlib import Path

In [11]:
df = pd.read_excel('../data/processed/one_year_mine_data_cleaned.xlsx')
df.head()

,Date,PM10,PM2.5,Temperature,Relative_humidity,RF-CUMULATIVE,Wind_speed,LOCATION,SEASON
0,2025-03-04,70.8,24.1,36.1,20.4,0.0,8.0,Near Office,Summer
1,2025-03-08,69.5,32.6,36.0,22.4,0.0,8.0,Near Office,Summer
2,2025-03-11,65.2,30.4,35.5,21.8,0.0,8.0,Near Office,Summer
3,2025-03-15,76.8,26.5,34.8,38.5,0.0,6.0,Near Office,Summer
4,2025-03-18,58.4,20.4,35.1,31.4,0.0,7.0,Near Office,Summer


<h2 align="center">Shapiro-Wilk test for normality</h2>
<p align="center">Tests whether a dataset is drawn from a normal distribution</p>

In [12]:
for column in [
    "Temperature",
    "Relative_humidity",
    "RF-CUMULATIVE",
    "Wind_speed",
    "PM10",
    "PM2.5"
]:

    stat, p = shapiro(df[column].sample(500, random_state=42))

    print(column)

    print(f"P-value : {p:.4f}")

    print()

Temperature
P-value : 0.0000

Relative_humidity
P-value : 0.0000

RF-CUMULATIVE
P-value : 0.0000

Wind_speed
P-value : 0.0000

PM10
P-value : 0.0426

PM2.5
P-value : 0.0000



<h2 align="center">Pearson correlation coefficient</h2>
<p align="center">The coefficient measures the linear relationship between two datasets, ranging from -1 (perfect negative correlation) to +1 (perfect positive correlation), with 0 indicating no linear correlation.</p>

In [13]:
weather = [
    "Temperature",
    "Relative_humidity",
    "RF-CUMULATIVE",
    "Wind_speed"
]

for feature in weather:

    r, p = pearsonr(df[feature], df["PM10"])

    print(f"{feature}")

    print(f"Correlation : {r:.3f}")

    print(f"P-value : {p:.4f}")

    print()

Temperature
Correlation : 0.118
P-value : 0.0046

Relative_humidity
Correlation : -0.087
P-value : 0.0366

RF-CUMULATIVE
Correlation : -0.039
P-value : 0.3512

Wind_speed
Correlation : 0.134
P-value : 0.0013



<h2 align="center">Spearman rank-order correlation coefficient</h2>
<p align="center">This nonparametric measure evaluates the monotonic relationship between two datasets, with values ranging from -1 (perfect negative correlation) to +1 (perfect positive correlation).</p>

In [14]:
weather = [
    "Temperature",
    "Relative_humidity",
    "RF-CUMULATIVE",
    "Wind_speed"
]

for feature in weather:

    r, p = spearmanr(df[feature], df["PM10"])

    print(f"{feature}")

    print(f"Correlation : {r:.3f}")

    print(f"P-value : {p:.4f}")

    print()

Temperature
Correlation : 0.107
P-value : 0.0100

Relative_humidity
Correlation : -0.073
P-value : 0.0820

RF-CUMULATIVE
Correlation : -0.075
P-value : 0.0716

Wind_speed
Correlation : 0.117
P-value : 0.0048



In [15]:
X = df[
    [
        "Temperature",
        "Relative_humidity",
        "RF-CUMULATIVE",
        "Wind_speed"
    ]
]
y = df["PM10"]
X = sm.add_constant(X)

In [16]:
model = sm.OLS(y, X).fit()

In [17]:
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                   PM10   R-squared:                       0.019
Model:                            OLS   Adj. R-squared:                  0.012
Method:                 Least Squares   F-statistic:                     2.801
Date:                Mon, 06 Jul 2026   Prob (F-statistic):             0.0253
Time:                        14:59:25   Log-Likelihood:                -2123.4
No. Observations:                 576   AIC:                             4257.
Df Residuals:                     571   BIC:                             4278.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                49.7045     10.45

In [19]:
X1 = df[
    [
        "Temperature",
        "Relative_humidity",
        "RF-CUMULATIVE",
        "Wind_speed"
    ]
]
X1 = sm.add_constant(X)
y1 = pd.to_numeric(df["PM2.5"], errors="coerce")
valid_idx = y1.notna() & X1.notna().all(axis=1)

In [20]:
model_pm25 = sm.OLS(y1.loc[valid_idx], X1.loc[valid_idx]).fit()
print(model_pm25.summary())

                            OLS Regression Results                            
Dep. Variable:                  PM2.5   R-squared:                       0.048
Model:                            OLS   Adj. R-squared:                  0.042
Method:                 Least Squares   F-statistic:                     7.224
Date:                Mon, 06 Jul 2026   Prob (F-statistic):           1.12e-05
Time:                        15:00:15   Log-Likelihood:                -1934.5
No. Observations:                 576   AIC:                             3879.
Df Residuals:                     571   BIC:                             3901.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                13.1371      7.53

In [21]:
results = pd.DataFrame({
    "Coefficient": model.params,
    "P-value": model.pvalues,
    "Standard Error": model.bse,
    "t-value": model.tvalues
})

display(results)

,Coefficient,P-value,Standard Error,t-value
const,49.704477,0.000003,10.457078,4.753190
Temperature,0.222622,0.473095,0.310090,0.717928
Relative_humidity,0.001991,0.957838,0.037646,0.052890
RF-CUMULATIVE,0.006721,0.595864,0.012665,0.530655
Wind_speed,0.550776,0.104844,0.339066,1.624393


In [22]:
Path("../data/processed").mkdir(parents=True, exist_ok=True)
results.to_excel(
    "../data/processed/stat_results2.xlsx",
    index=False
)